In [1]:
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "db" / "retail.db"
REPORTS_DIR = PROJECT_ROOT / "reports"
SEED = 42

conn = sqlite3.connect(DB_PATH)

tx = pd.read_sql("""
SELECT t.transaction_id, t.order_id, t.payment_method, t.amount, t.transaction_ts,
       o.store_id, o.channel,
       ot.order_total, ot.total_quantity
FROM clean_transactions AS t
JOIN clean_orders AS o ON o.order_id = t.order_id
JOIN (SELECT order_id,
             SUM(line_revenue) AS order_total,
             SUM(quantity) AS total_quantity
      FROM clean_order_items
      GROUP BY order_id) AS ot ON ot.order_id = t.order_id
WHERE t.status = 'Success'
""", conn, parse_dates=["transaction_ts"])

print(len(tx))
tx.head(3)

81874


,transaction_id,order_id,payment_method,amount,transaction_ts,store_id,channel,order_total,total_quantity
0,T00000001,O0000001,UPI,1640.0,2024-01-02 16:36:31,S004,Online,1640.0,1
1,T00000002,O0000002,UPI,15520.5,2024-01-03 11:59:52,S002,In-store,15520.5,7
2,T00000003,O0000003,Card,4920.0,2024-01-03 18:36:26,S005,In-store,4920.0,3


In [2]:
tx["hour"] = tx["transaction_ts"].dt.hour
tx["amount_ratio"] = (tx["amount"] / tx["order_total"]).round(3)
tx["payments_for_order"] = tx.groupby("order_id")["transaction_id"].transform("count")

tx[["amount", "amount_ratio", "hour", "payments_for_order", "total_quantity"]].describe().round(2)

,amount,amount_ratio,hour,payments_for_order,total_quantity
count,81874.00,81874.00,81874.00,81874.00,81874.00
mean,8386.68,1.00,14.57,1.01,4.19
std,10898.08,0.25,3.38,0.17,2.71
min,28.00,1.00,1.00,1.00,1.00
25%,2730.00,1.00,12.00,1.00,2.00
50%,5591.00,1.00,15.00,1.00,4.00
75%,10725.88,1.00,17.00,1.00,6.00
max,1437940.00,37.33,23.00,5.00,118.00


In [3]:
q1, q3 = tx["total_quantity"].quantile([0.25, 0.75])
QUANTITY_LIMIT = q3 + 3 * (q3 - q1)
OPEN_HOUR = 8

rules = {
    "large_basket": tx["total_quantity"] > QUANTITY_LIMIT,
    "outside_store_hours": tx["hour"] < OPEN_HOUR,
    "repeated_payment": tx["payments_for_order"] > 1,
    "amount_mismatch": (tx["amount_ratio"] - 1).abs() > 0.05,
}

for name, hit in rules.items():
    tx[f"rule_{name}"] = hit
    print(f"{name:<22} {hit.sum():>4} transactions")

print("\nLarge-basket limit:", QUANTITY_LIMIT, "items")

large_basket             30 transactions
outside_store_hours      30 transactions
repeated_payment        150 transactions
amount_mismatch          73 transactions

Large-basket limit: 18.0 items


In [4]:
MODEL_FEATURES = ["amount", "amount_ratio", "hour", "payments_for_order", "total_quantity"]

X = tx[MODEL_FEATURES].copy()
X["amount"] = np.log1p(X["amount"])

iso = IsolationForest(n_estimators=300, contamination=0.001, random_state=SEED)
iso.fit(X)

tx["anomaly_score"] = (-iso.score_samples(X)).round(3)
tx["rule_isolation_forest"] = iso.predict(X) == -1

print("Flagged by Isolation Forest:", tx["rule_isolation_forest"].sum())
tx.sort_values("anomaly_score", ascending=False)[
    ["transaction_id", "amount", "amount_ratio", "hour", "payments_for_order", "total_quantity", "anomaly_score"]
].head(8)

Flagged by Isolation Forest: 82


,transaction_id,amount,amount_ratio,hour,payments_for_order,total_quantity,anomaly_score
60579,T00062458,153860.0,1.0,22,1,92,0.719
77616,T00080006,399172.5,1.0,8,1,92,0.713
64138,T00066104,127270.0,1.0,8,1,67,0.712
10607,T00010942,35689.5,1.0,22,1,11,0.701
30122,T00031078,55162.0,1.0,22,1,10,0.700
58241,T00060049,52866.0,1.0,21,1,11,0.698
79695,T00082142,36.0,1.0,8,1,1,0.697
63316,T00065262,179820.0,1.0,19,1,109,0.697


In [5]:
def explain(row):
    reasons = []
    if row["rule_large_basket"]:
        reasons.append(f"{int(row['total_quantity'])} items in one order (normal max about {int(QUANTITY_LIMIT)})")
    if row["rule_outside_store_hours"]:
        reasons.append(f"paid at {row['hour']:02d}:00, before stores open ({OPEN_HOUR}:00)")
    if row["rule_repeated_payment"]:
        reasons.append(f"{int(row['payments_for_order'])} successful payments for the same order")
    if row["rule_amount_mismatch"]:
        reasons.append(f"paid Rs {row['amount']:,.0f} but order items total Rs {row['order_total']:,.0f}")
    if row["rule_isolation_forest"] and not reasons:
        reasons.append("unusual combination of amount, time and basket size (Isolation Forest)")
    return "; ".join(reasons)

def priority(row):
    if row["rules_triggered"] >= 2 or row["rule_amount_mismatch"] or row["rule_repeated_payment"]:
        return "High"
    if row["rule_large_basket"] or row["rule_outside_store_hours"]:
        return "Medium"
    return "Low"

rule_columns = [c for c in tx.columns if c.startswith("rule_")]
flagged = tx[tx[rule_columns].any(axis=1)].copy()

flagged["rules_triggered"] = flagged[rule_columns].sum(axis=1)
flagged["reason"] = flagged.apply(explain, axis=1)
flagged["priority"] = flagged.apply(priority, axis=1)

print("Transactions flagged:", len(flagged))
print(flagged["priority"].value_counts())
flagged[["transaction_id", "priority", "reason"]].head(8)

Transactions flagged: 349
priority
High      238
Low        66
Medium     45
Name: count, dtype: int64


,transaction_id,priority,reason
742,T00000776,Medium,"paid at 02:00, before stores open (8:00)"
837,T00000872,High,"paid Rs 167,042 but order items total Rs 13,048"
897,T00000933,Medium,"paid at 02:00, before stores open (8:00)"
1297,T00001352,High,5 successful payments for the same order
2369,T00002468,Low,"unusual combination of amount, time and basket..."
2406,T00002505,Medium,"paid at 03:00, before stores open (8:00)"
3463,T00003594,High,"paid Rs 18,364 but order items total Rs 13,468"
3535,T00003668,Low,"unusual combination of amount, time and basket..."


In [6]:
MODEL_VERSION = "anomaly_rules_iforest_v1"
flagged_at = datetime.now(timezone.utc).isoformat(timespec="seconds")
DISCLAIMER = ("An anomaly flag or score means this transaction is unusual compared with others. "
              "It is NOT proof of fraud or wrongdoing and must be reviewed by a person.")

anomaly_flags = flagged[["transaction_id", "order_id", "store_id", "transaction_ts", "payment_method",
                         "amount", "order_total", "anomaly_score", "rules_triggered",
                         "priority", "reason"]].copy()
anomaly_flags["transaction_ts"] = anomaly_flags["transaction_ts"].dt.strftime("%Y-%m-%d %H:%M:%S")
anomaly_flags["review_status"] = "Pending review"
anomaly_flags["reviewed_by"] = None
anomaly_flags["review_note"] = None
anomaly_flags["model_version"] = MODEL_VERSION
anomaly_flags["flagged_at"] = flagged_at
anomaly_flags["disclaimer"] = DISCLAIMER

priority_order = {"High": 0, "Medium": 1, "Low": 2}
anomaly_flags = anomaly_flags.sort_values(
    by=["priority", "anomaly_score"],
    key=lambda col: col.map(priority_order) if col.name == "priority" else -col,
)

anomaly_flags.to_sql("anomaly_flags", conn, if_exists="replace", index=False)
anomaly_flags.to_csv(REPORTS_DIR / "anomaly_flags.csv", index=False)

print("Saved anomaly_flags:", len(anomaly_flags), "rows")
anomaly_flags.head(5)

Saved anomaly_flags: 349 rows


,transaction_id,order_id,store_id,transaction_ts,payment_method,amount,order_total,anomaly_score,rules_triggered,priority,reason,review_status,reviewed_by,review_note,model_version,flagged_at,disclaimer
60579,T00062458,O0060529,S010,2025-08-19 22:29:30,UPI,153860.0,153860.0,0.719,2,High,92 items in one order (normal max about 18),Pending review,None,None,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00,An anomaly flag or score means this transactio...
77616,T00080006,O0077700,S007,2025-12-02 08:08:36,Card,399172.5,399172.5,0.713,2,High,92 items in one order (normal max about 18),Pending review,None,None,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00,An anomaly flag or score means this transactio...
64138,T00066104,O0064225,S006,2025-09-15 08:44:41,UPI,127270.0,127270.0,0.712,2,High,67 items in one order (normal max about 18),Pending review,None,None,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00,An anomaly flag or score means this transactio...
51236,T00052841,O0051232,S005,2025-06-08 10:11:58,Card,109952.5,109952.5,0.697,2,High,73 items in one order (normal max about 18),Pending review,None,None,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00,An anomaly flag or score means this transactio...
63316,T00065262,O0063314,S005,2025-09-08 19:13:58,Wallet,179820.0,179820.0,0.697,2,High,109 items in one order (normal max about 18),Pending review,None,None,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00,An anomaly flag or score means this transactio...


In [7]:
planted = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "_answer_key" / "planted_anomalies.csv")

flagged["base_transaction_id"] = flagged["transaction_id"].str.replace(r"R\d+$", "", regex=True)
planted["caught"] = planted["transaction_id"].isin(flagged["base_transaction_id"])

planted.groupby("type")["caught"].agg(caught="sum", planted="count")

,caught,planted
type,,
amount_mismatch,30,30
huge_amount,30,30
odd_hour,30,30
rapid_repeat,30,30


In [8]:
is_planted = flagged["base_transaction_id"].isin(planted["transaction_id"])
not_planted = flagged[~is_planted]

raw_items = pd.read_sql("SELECT order_id, quantity FROM raw_order_items", conn)
orders_with_removed_lines = set(raw_items.loc[raw_items["quantity"].astype(int) <= 0, "order_id"])

cleaning_related = not_planted["rule_amount_mismatch"] & not_planted["order_id"].isin(orders_with_removed_lines)
model_only = not_planted["rule_isolation_forest"] & (not_planted["rules_triggered"] == 1)

print("Flags not in the answer key:", len(not_planted))
print("  Amount mismatch caused by lines removed in cleaning:", cleaning_related.sum())
print("  Flagged by Isolation Forest only:", model_only.sum())

Flags not in the answer key: 109
  Amount mismatch caused by lines removed in cleaning: 43
  Flagged by Isolation Forest only: 66


In [9]:
summary = pd.DataFrame([
    {"metric": "Successful transactions checked", "value": len(tx)},
    {"metric": "Transactions flagged", "value": len(flagged)},
    {"metric": "Flagged share (%)", "value": round(100 * len(flagged) / len(tx), 2)},
    {"metric": "High priority flags", "value": int((flagged["priority"] == "High").sum())},
    {"metric": "Medium priority flags", "value": int((flagged["priority"] == "Medium").sum())},
    {"metric": "Low priority flags", "value": int((flagged["priority"] == "Low").sum())},
    {"metric": "Planted anomalies caught (of 120)", "value": int(planted["caught"].sum())},
    {"metric": "Mismatches explained by cleaning", "value": int(cleaning_related.sum())},
    {"metric": "Isolation Forest-only flags", "value": int(model_only.sum())},
])
summary["model_version"] = MODEL_VERSION
summary["created_at"] = flagged_at

summary.to_sql("anomaly_summary", conn, if_exists="replace", index=False)
summary.to_csv(REPORTS_DIR / "anomaly_summary.csv", index=False)

summary

,metric,value,model_version,created_at
0,Successful transactions checked,81874.00,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00
1,Transactions flagged,349.00,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00
2,Flagged share (%),0.43,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00
3,High priority flags,238.00,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00
4,Medium priority flags,45.00,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00
5,Low priority flags,66.00,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00
6,Planted anomalies caught (of 120),120.00,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00
7,Mismatches explained by cleaning,43.00,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00
8,Isolation Forest-only flags,66.00,anomaly_rules_iforest_v1,2026-11-02T09:43:48+00:00


In [10]:
conn.close()